# Spick V10 Code

### Setup: Prüfungsdaten einlesen und Tabellen prüfen
- Alle drei Dateien der Musterprüfung laden
- `df` enthält die Reviews, `df_runs` die simulierten Modellantworten und `df_models` die Daten für den Modellvergleich
- Das Dateiformat bestimmt den Lesebefehl. Für CSV mit Komma verwendest du `pd.read_csv(pfad)`, für CSV mit Semikolon `pd.read_csv(pfad, sep=";")`
- Für HTML `pd.read_html(pfad)`

In [16]:
# 1. Bibliotheken laden
from pathlib import Path                                                        #Erstellt Pfade zu Ordnern und Dateien
import pandas as pd                                                             #Liest Dateien ein und verarbeitet Tabellen
from sklearn.metrics import accuracy_score                                      #Berechnet später den Anteil korrekter Vorhersagen

# 2. Ordner mit den Prüfungsdaten bestimmen
EXAM_DIR = Path.cwd()                                                           #Startet im aktuellen Arbeitsordner
if not (EXAM_DIR / "reviews.csv").exists():                                     #Fehlt die Datei dort, verwende den Unterordner der Musterprüfung
    EXAM_DIR = EXAM_DIR / "08_Example_Exam_Questions"

# 3. Jede Datei als eigene Tabelle einlesen
df = pd.read_csv(EXAM_DIR / "reviews.csv")
df_runs = pd.read_csv(EXAM_DIR / "llm_runs.csv")
df_models = pd.read_csv(EXAM_DIR / "model_comparison.csv")

# 4. Vorgaben der Musterprüfung übernehmen
df["stars_label"] = df["stars"].map({                                           #Übersetzt die Sterne jedes Reviews in ein Label
    1: "negative", 2: "negative", 3: "mixed", 4: "positive", 5: "positive"
})
PRICE_INPUT_PER_M = 0.10                                                        #Vorgegebener Preis in USD pro 1 Million Input-Tokens

# 5. Grösse, Spaltennamen und zwei Beispielzeilen jeder Tabelle anzeigen
print("df – Reviews:", df.shape)
display(df.head(2))

print("df_runs – Modellantworten:", df_runs.shape)
display(df_runs.head(2))

print("df_models – Modellvergleich:", df_models.shape)
display(df_models.head(2))

df – Reviews: (300, 8)


,review_id,store,date,stars,text,n_tokens,gold_label,stars_label
0,R001,Basel,2025-06-30,1,Came here on a Tuesday afternoon. The quality ...,32,negative,negative
1,R002,Luzern,2026-06-09,3,I have been shopping here for years. Nobody in...,36,mixed,mixed


df_runs – Modellantworten: (1500, 3)


,review_id,run,label
0,R001,1,negative
1,R001,2,negative


df_models – Modellvergleich: (120, 6)


,model,review_id,label,seconds,input_tokens,output_tokens
0,gpt-6-luna,R255,positive,1.33,210,87
1,qwen3:1.7b,R255,positive,10.22,210,196


### Tokens zählen und Kosten berechnen
Der Codeblock zählt die Tokens aller Reviewtexte, addiert die Prompt-Tokens für jeden Aufruf und berechnet daraus die erwarteten Input-Kosten in USD

- Eine Zeile in `df` entspricht einem Review. Die Spalte `n_tokens` enthält die Anzahl Tokens seines Texts. Für «alle Reviews zusammen» `.sum()` verwenden, für «im Durchschnitt pro Review» `.mean()`
- Achten, wie oft der Prompt mitgeschickt wird. Bei einem Aufruf pro Review ist `len(df)` die Anzahl Aufrufe. Die gesamten Input-Tokens ergeben sich aus den Review-Tokens plus den Prompt-Tokens aller Aufrufe
- Für die Kosten zählt die Preiseinheit. Bei einem Preis pro 1 Million Tokens teilt man die Tokenzahl durch `1_000_000` und multipliziert mit dem Preis. Gibt die Aufgabe die Tokenzahl bereits vor, diese direkt einsetzen
- Bei doppelter Prompt-Länge verdoppelt sich der Kostenanteil des Prompts. Die Tokens der Reviewtexte bleiben gleich, daher steigen die gesamten Input-Kosten auf weniger als das Doppelte
- Für «Wie viele Input-Tokens oder Kosten entstehen je Filiale?» zuerst pro Review den Prompt dazurechnen und danach mit `groupby("store")` die Tokens jeder Filiale summieren

In [29]:
#1. Tokens der Reviewtexte zusammenzählen
review_tokens = df["n_tokens"].sum()                                            #Addiert die Tokenzahlen aller Reviews
print("Tokens aller Reviewtexte:", review_tokens)
print("Tokens pro Review im Durchschnitt:", df["n_tokens"].mean())

#2. Prompt für jeden Aufruf mitzählen
PROMPT_TOKENS = 185                                                             #Anzahl Tokens des Prompts aus der Aufgabe übernehmen
anzahl_aufrufe = len(df)                                                        #Hier wird jeder Review genau einmal gesendet
input_tokens = review_tokens + anzahl_aufrufe * PROMPT_TOKENS
print("Input-Tokens inkl. Prompts:", input_tokens)

#3. Kosten berechnen
tokens_fuer_kosten = input_tokens                                               #Bei vorgegebener Tokenzahl hier z. B. 250_000 einsetzen
kosten_usd = tokens_fuer_kosten / 1_000_000 * PRICE_INPUT_PER_M
print("Input-Kosten in USD:", round(kosten_usd, 4))                             #Rundet die Ausgabe auf vier Nachkommastellen

# 4. Input-Tokens und Kosten je Filiale berechnen
df["input_tokens"] = df["n_tokens"] + PROMPT_TOKENS                             #Pro Review die Tokens seines Texts und eines Prompts addieren
tokens_je_filiale = df.groupby("store")["input_tokens"].sum()                   #Reviews nach Filiale gruppieren und deren Input-Tokens je Filiale addieren
kosten_je_filiale = tokens_je_filiale / 1_000_000 * PRICE_INPUT_PER_M           #Tokenzahl jeder Filiale in USD umrechnen

print("Input-Tokens je Filiale:", tokens_je_filiale, sep="\n")
print("Kosten in USD je Filiale:", kosten_je_filiale.round(4), sep="\n")

Tokens aller Reviewtexte: 10330
Tokens pro Review im Durchschnitt: 34.43333333333333
Input-Tokens inkl. Prompts: 65830
Input-Kosten in USD: 0.0066
Input-Tokens je Filiale:
store
Basel     14867
Bern      16519
Luzern    16617
Zürich    17827
Name: input_tokens, dtype: int64
Kosten in USD je Filiale:
store
Basel     0.0015
Bern      0.0017
Luzern    0.0017
Zürich    0.0018
Name: input_tokens, dtype: float64


### Reviews klassifizieren: Prompt und JSON-Schema prüfen
Codeblock legt fest, wie das Modell die Stimmung eines Reviews beurteilen soll und welche Antworten im JSON-Format erlaubt sind, ohne einen API-Aufruf auszuführen
- Im Prompt festlegen, was das Modell beurteilen soll. Für die Stimmung (`sentiment`) den Reviewtext als Grundlage verwenden. Die Sternebewertung kann vom Text abweichen
- Im JSON-Schema den Feldnamen und die erlaubten Antworten prüfen. Der Feldname muss zum Prompt passen. Die Liste `enum` muss genau die in der Aufgabe verlangten Kategorien enthalten
- Für die erwartete JSON-Antwort den Reviewtext lesen und das passende Label wählen. Bsp. aus der Musterprüfung: Lob für den bequemen Stuhl und Kritik am beschädigten Bein ergeben `{"sentiment": "mixed"}`

In [18]:
#1. Prompt festlegen: Was beurteilen und wie antworten?
instructions_B2 = (
    "Classify the sentiment using only the review text. "
    "Use positive for praise, negative for criticism and mixed for both. "
    "Return a JSON object with the field sentiment."
)

#2. JSON-Schema festlegen: Welches Feld und welche Werte sind erlaubt?
schema_B2 = {
    "type": "object",
    "properties": {                                                             #Beschreibt die Felder der Antwort
        "sentiment": {                                                          #Feldname aus dem Prompt übernehmen
            "type": "string",                                                   #Der Wert muss Text sein
            "enum": ["positive", "negative", "mixed"]                           #Für das Feld sentiment ist genau einer dieser drei Werte erlaubt
        }
    },
    "required": ["sentiment"],                                                  #Jede Antwort muss das Feld sentiment enthalten
    "additionalProperties": False                                               #Zusätzliche Felder ausschliessen
}

#3. Erlaubte Labels zur Kontrolle anzeigen
print(schema_B2["properties"]["sentiment"]["enum"])

['positive', 'negative', 'mixed']


### Prüfen, ob das Modell bei mehreren Durchläufen dasselbe Label vergibt
Codeblock vergleicht die gespeicherten Modellantworten für jeden Review und berechnet, bei wie vielen Reviews das Label wechselt

- Mit `pivot()` die Labels nebeneinander anordnen. Danach steht jede Zeile für einen Review und jede Spalte für einen Durchlauf. Liegt die Tabelle bereits so vor, direkt mit `nunique(axis=1)` beginnen
- `nunique(axis=1)` zählt die verschiedenen Labels innerhalb einer Zeile. `> 1` erkennt wechselnde Labels, `== 1` erkennt gleichbleibende Labels
- Fünfmal `negative` zeigt, dass das Modell für diesen Review konsistent antwortet. Für die inhaltliche Richtigkeit das Ergebnis mit dem geprüften Referenzlabel (Gold Label) vergleichen. Reviews mit wechselnden Labels manuell prüfen

In [19]:
# 1. Gespeicherte Labels pro Review nebeneinander anordnen
runs_C1 = df_runs.pivot(index="review_id", columns="run", values="label")

# 2. Jeden Review mit True markieren, wenn mindestens zwei verschiedene Labels vorkommen
changed_C1 = runs_C1.nunique(axis=1) > 1

# 3. Anzahl oder Anteil ausgeben, je nach Prüfungsfrage
print("Anzahl Reviews mit wechselndem Label:", changed_C1.sum())                #Zählt alle True
print("Anteil Reviews mit wechselndem Label:", round(changed_C1.mean(), 3))     #0.227 bedeutet 22.7%

Anzahl Reviews mit wechselndem Label: 68
Anteil Reviews mit wechselndem Label: 0.227


### Belegzitate mit dem ursprünglichen Reviewtext vergleichen
Codeblock prüft für jedes vorgegebene Zitat, ob es wörtlich im zugehörigen Reviewtext vorkommt, und berechnet den Anteil der gefundenen Zitate
- `df` ist die bereits geladene Tabelle mit den vollständigen Reviews. `review_id` kennzeichnet den Review, `text` enthält seinen Originaltext
- `quotes_C2` ist eine zweite Tabelle mit den vorgegebenen Zitaten aus der Musterprüfung. Jede Zeile enthält die ID eines Reviews und ein angeblich wörtliches Zitat (`evidence`). `R001` kommt zweimal vor, weil zwei Zitate aus demselben Review geprüft werden
- `df_C2` verbindet beide Tabellen über die `review_id`. Dadurch stehen das Zitat und der passende Originaltext in derselben Zeile. `found` zeigt nach der Prüfung für jedes Zitat `True` für gefunden oder `False` für nicht gefunden

In [21]:
#1. Zitate erfassen. Jede innere Klammer entspricht einer Tabellenzeile
quotes_C2 = pd.DataFrame([
    ("R001", "Came here on a Tuesday afternoon"),
    ("R001", "came here on a tuesday afternoon"),
    ("R002", "I have been shopping here for years"),
    ("R002", "I been shopping here for years"),
], columns=["review_id", "evidence"])                                           #Erster Wert = Review-ID, zweiter Wert = Zitat

#2. Neben jedes Zitat den vollständigen Text des zugehörigen Reviews setzen
df_C2 = quotes_C2.merge(df[["review_id", "text"]], on="review_id")

#3. Für jede Zeile prüfen, ob ihr Zitat genau so in ihrem Originaltext steht
found_C2 = []                                                                   #Sammelt ein Ergebnis pro Zitat
for _, row in df_C2.iterrows():                                                 #row ist die aktuelle Zeile, _ steht für die unbenutzte Zeilennummer
    found = row["evidence"] in row["text"]                                      #Sucht das Zitat im Originaltext, inkl. Gross- und Kleinschreibung
    found_C2.append(found)                                                      #Hängt True oder False an die Ergebnisliste an

#4. Ergebnisse als neue Spalte speichern und anzeigen
df_C2["found"] = found_C2
display(df_C2[["review_id", "evidence", "found"]])
print("Anteil gefundener Zitate:", df_C2["found"].mean())                       #Bsp. 2 von 4 gefunden ergibt 0.5

,review_id,evidence,found
0,R001,Came here on a Tuesday afternoon,True
1,R001,came here on a tuesday afternoon,False
2,R002,I have been shopping here for years,True
3,R002,I been shopping here for years,False


Anteil gefundener Zitate: 0.5


### Modellvorhersagen prüfen und Genauigkeit (Accuracy) vergleichen
Der Codeblock vergleicht für jedes Modell die vorhergesagten Labels mit den geprüften Referenzlabels (Gold Labels) und berechnet den Anteil richtiger Vorhersagen
- `df_models` enthält die Vorhersagen. In `model` steht der Modellname, in `label` die Vorhersage. `df` enthält in `gold_label` die richtige Antwort. Beide Tabellen über `review_id` verbinden, damit Vorhersage und richtige Antwort zum selben Review gehören
- Eine Accuracy von `0.967` bedeutet, dass das Modell 96.7% der geprüften Reviews richtig klassifiziert hat. Beim Modellvergleich zusätzlich auf Sekunden pro Review, API-Kosten und Vorgaben zur lokalen Verarbeitung achten
- Für «Welche Labels verwechselt das Modell?» eine Kreuztabelle verwenden. Die Zeilen zeigen die richtigen Labels, die Spalten die Vorhersagen. Jede Zahl gibt an, wie viele Reviews diese Kombination haben
- Für «Wie oft vergibt das Modell jedes Label?» die Vorhersagen mit `.value_counts()` zählen. Mit `.value_counts(normalize=True)` erhält man die Anteile
- Stehen die Vorhersagen zweier Prompts nebeneinander in den Spalten `vague` und `precise`, die Accuracy für jede Spalte einzeln berechnen. Für den ersten Prompt `accuracy_score(test["silver"], test["vague"])` verwenden, für den zweiten `accuracy_score(test["silver"], test["precise"])`
- `test` ist die Tabelle aus der Übung. Die Spalte `silver` enthält die vom Modell erzeugten Referenzlabels (Silver Labels). Heisst die Referenzspalte in der Prüfung `gold_label`, in beiden Aufrufen `test["silver"]` durch `test["gold_label"]` ersetzen

In [30]:
# 1. Jede Vorhersage über die review_id mit dem richtigen Label desselben Reviews verbinden
df_C3 = df_models.merge(df[["review_id", "gold_label"]], on="review_id")

# 2. Ein Modell nach dem anderen auswerten
for model in df_C3["model"].unique():                                           #model enthält den Namen des aktuellen Modells
    test_C3 = df_C3[df_C3["model"] == model]                                    #Nur die Vorhersagen dieses Modells auswählen

    # 3. Wie oft liegt dieses Modell richtig?
    accuracy = accuracy_score(test_C3["gold_label"], test_C3["label"])          #Referenzlabel und Vorhersage pro Review vergleichen
    print(model, f"{accuracy:.3f}")                                             #Modellname und Accuracy mit drei Nachkommastellen ausgeben

    # 4. Welche Labels verwechselt dieses Modell?
                                                                                #Zeile = Referenzlabel, Spalte = Vorhersage, Zahl = Anzahl Reviews mit dieser Kombination
    display(pd.crosstab(test_C3["gold_label"], test_C3["label"]))

    # 5. Wie oft vergibt dieses Modell jedes Label?
    print(test_C3["label"].value_counts())                                      #Zählt die Vorhersagen je Label für dieses Modell

gpt-6-luna 0.967


label,mixed,negative,positive
gold_label,,,
mixed,11,0,2
negative,0,18,0
positive,0,0,29


label
positive    31
negative    18
mixed       11
Name: count, dtype: int64
qwen3:1.7b 0.867


label,mixed,negative,positive
gold_label,,,
mixed,9,0,4
negative,2,16,0
positive,2,0,27


label
positive    31
negative    16
mixed       13
Name: count, dtype: int64


### Falsch klassifizierte Reviews finden
Der Codeblock zeigt pro Modell die falsch klassifizierten Reviews mit Reviewtext, Vorhersage (`label`) und geprüftem Referenzlabel (`gold_label`)
- `!=` bedeutet «ungleich». Eine Vorhersage gilt als falsch, wenn ihr Label vom Referenzlabel desselben Reviews abweicht. Für richtige Vorhersagen `==` verwenden
- Ein Review kann mehrfach erscheinen, wenn mehrere Modelle denselben Review falsch klassifiziert haben

In [ ]:
#Vorhersagen aus df_models mit Reviewtext und Referenzlabel aus df verbinden
vergleich = df_models.merge(df[["review_id", "text", "gold_label"]], on="review_id")

#Nur Zeilen behalten, in denen das Modell ein anderes Label als das Referenzlabel vergeben hat
fehler = vergleich[vergleich["label"] != vergleich["gold_label"]]

#Anzahl falscher Vorhersagen ausgeben und die betroffenen Reviews anzeigen
print("Anzahl falscher Vorhersagen:", len(fehler))
display(fehler[["model", "review_id", "text", "label", "gold_label"]])

#Variante: Reviews anzeigen, bei denen Sterne und Textstimmung voneinander abweichen
abweichungen = df[df["stars_label"] != df["gold_label"]]
display(abweichungen[["review_id", "stars", "stars_label", "text", "gold_label"]])

Anzahl falscher Vorhersagen: 10


,model,review_id,text,label,gold_label
26,gpt-6-luna,R109,Came here on a Tuesday afternoon. The delivery...,positive,mixed
27,qwen3:1.7b,R109,Came here on a Tuesday afternoon. The delivery...,positive,mixed
31,qwen3:1.7b,R221,The bed was easy to assemble and feels solid. ...,mixed,positive
40,gpt-6-luna,R092,Came here on a Tuesday afternoon. There was no...,positive,mixed
41,qwen3:1.7b,R092,Came here on a Tuesday afternoon. There was no...,positive,mixed
59,qwen3:1.7b,R105,We went to WohnWerk Bern to furnish our new fl...,positive,mixed
65,qwen3:1.7b,R142,I have been shopping here for years. The retur...,mixed,positive
89,qwen3:1.7b,R102,First visit to WohnWerk Zürich. Our table was ...,mixed,negative
99,qwen3:1.7b,R200,The restaurant ran out of almost every dish by...,positive,mixed
109,qwen3:1.7b,R279,There was no staff around to answer a simple q...,mixed,negative


,review_id,stars,stars_label,text,gold_label
3,R004,3,mixed,First visit to WohnWerk Basel. The employee at...,negative
6,R007,4,positive,They delivered the sofa a day earlier than pla...,mixed
10,R011,2,negative,Assembly took four hours because the holes did...,mixed
15,R016,3,mixed,I have been shopping here for years. An employ...,positive
17,R018,3,mixed,I have been shopping here for years. The retur...,positive
20,R021,4,positive,Came here on a Tuesday afternoon. The instruct...,mixed
29,R030,2,negative,I have been shopping here for years. Returning...,mixed
34,R035,5,positive,"Brilliant, the restaurant ran out of food at n...",negative
37,R038,2,negative,Visited the Zürich store last weekend. The cas...,mixed
43,R044,2,negative,We went to WohnWerk Basel to furnish our new f...,mixed


### Modelle vergleichen: Antwortzeit und API-Kosten pro 10'000 Reviews

Der Codeblock berechnet aus den gespeicherten Ergebnissen die durchschnittliche Antwortzeit pro Review und schätzt für jedes Modell die API-Kosten für 10'000 Reviews
- `df_models` enthält eine Zeile pro Modell und Review. `seconds` enthält die Antwortzeit in Sekunden. `input_tokens` enthält bereits Prompt und Reviewtext. `output_tokens` enthält die Tokens der Antwort inkl. Reasoning-Tokens
- Für die Kostenschätzung zuerst die durchschnittlichen Input- und Output-Tokens pro Review mit ihren jeweiligen Preisen verrechnen. Danach mit der gefragten Anzahl Reviews multiplizieren. Die Schätzung setzt einen ähnlichen Tokenverbrauch wie im Sample voraus

In [ ]:
#1. Durchschnitt pro Modell berechnen. Jede Ergebniszeile steht für ein Modell
vergleich = df_models.groupby("model")[["seconds", "input_tokens", "output_tokens"]].mean()

#2. Vorgaben aus der Aufgabe übernehmen
anzahl_reviews = 10_000
preis_input = 0.10                                                              #USD pro 1 Million Input-Tokens
preis_output = 0.50                                                             #USD pro 1 Million Output-Tokens

#3. API-Kosten berechnen. Beim lokalen Modell bleiben sie bei 0 USD
vergleich["api_kosten_usd"] = 0.0
api_modell = "gpt-6-luna"                                                       #Name des API-Modells aus der Spalte model
api_werte = vergleich.loc[api_modell]                                           #Durchschnittswerte dieses Modells auswählen

kosten_pro_review = (
    api_werte["input_tokens"] * preis_input
    + api_werte["output_tokens"] * preis_output
) / 1_000_000

vergleich.loc[api_modell, "api_kosten_usd"] = kosten_pro_review * anzahl_reviews

#4. Mittlere Sekunden pro Review und geschätzte USD für alle 10'000 Reviews anzeigen
display(vergleich[["seconds", "api_kosten_usd"]].round(3))

,seconds,api_kosten_usd
model,,
gpt-6-luna,1.247,0.551
qwen3:1.7b,8.030,0.000
